## API

    shop-frontend
    ├── dependsOn ──────> shop-api Component
    └── consumesApis ───> shop-api API
    
    shop-api Component
    ├── providesApis ───> shop-api API
    ├── dependsOn ──────> product-service
    ├── dependsOn ──────> order-service
    └── dependsOn ──────> payment-service

In [ ]:
%%bash
cat > ~/lemon-shop/examples/shop-api-definition.yaml <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: API
metadata:
  name: shop-api
  namespace: default
  title: Lemon Shop API
  description: Zentrale HTTP-API des Lemon Shops für Frontend und externe Clients
  tags:
    - lemon-shop
    - python
    - fastapi
    - rest
    - openapi
spec:
  type: openapi
  lifecycle: experimental
  owner: shop-team
  system: lemon-shop
  definition: |
    openapi: 3.0.3
    info:
      title: Lemon Shop API
      description: |
        Zentrale API für den Lemon Shop.

        Die API stellt Funktionen für Produkte, Bestellungen und Zahlungen
        über eine einheitliche Schnittstelle bereit.
      version: 0.1.0

    servers:
      - url: /api
        description: Relativer API-Endpunkt

    tags:
      - name: Health
        description: Status- und Verfügbarkeitsprüfungen
      - name: Products
        description: Produkte und Produktkatalog
      - name: Orders
        description: Warenkörbe und Bestellungen
      - name: Payments
        description: Zahlungsprozesse

    paths:
      /health:
        get:
          tags:
            - Health
          summary: Status der Shop API abrufen
          operationId: getHealth
          responses:
            '200':
              description: Die API ist verfügbar
              content:
                application/json:
                  schema:
                    $ref: '#/components/schemas/HealthResponse'

      /products:
        get:
          tags:
            - Products
          summary: Produkte auflisten
          operationId: listProducts
          responses:
            '200':
              description: Liste der verfügbaren Produkte
              content:
                application/json:
                  schema:
                    type: array
                    items:
                      $ref: '#/components/schemas/Product'

      /products/{productId}:
        get:
          tags:
            - Products
          summary: Einzelnes Produkt abrufen
          operationId: getProduct
          parameters:
            - name: productId
              in: path
              required: true
              schema:
                type: string
                format: uuid
          responses:
            '200':
              description: Produktdetails
              content:
                application/json:
                  schema:
                    $ref: '#/components/schemas/Product'
            '404':
              description: Produkt wurde nicht gefunden

      /orders:
        post:
          tags:
            - Orders
          summary: Neue Bestellung erstellen
          operationId: createOrder
          requestBody:
            required: true
            content:
              application/json:
                schema:
                  $ref: '#/components/schemas/CreateOrderRequest'
          responses:
            '201':
              description: Bestellung wurde erstellt
              content:
                application/json:
                  schema:
                    $ref: '#/components/schemas/Order'
            '400':
              description: Ungültige Bestelldaten

      /orders/{orderId}:
        get:
          tags:
            - Orders
          summary: Bestellung abrufen
          operationId: getOrder
          parameters:
            - name: orderId
              in: path
              required: true
              schema:
                type: string
                format: uuid
          responses:
            '200':
              description: Bestelldetails
              content:
                application/json:
                  schema:
                    $ref: '#/components/schemas/Order'
            '404':
              description: Bestellung wurde nicht gefunden

      /orders/{orderId}/payments:
        post:
          tags:
            - Payments
          summary: Zahlung für eine Bestellung auslösen
          operationId: createPayment
          parameters:
            - name: orderId
              in: path
              required: true
              schema:
                type: string
                format: uuid
          requestBody:
            required: true
            content:
              application/json:
                schema:
                  $ref: '#/components/schemas/CreatePaymentRequest'
          responses:
            '201':
              description: Zahlung wurde erstellt
              content:
                application/json:
                  schema:
                    $ref: '#/components/schemas/Payment'
            '400':
              description: Zahlung konnte nicht verarbeitet werden
            '404':
              description: Bestellung wurde nicht gefunden

    components:
      schemas:
        HealthResponse:
          type: object
          required:
            - status
          properties:
            status:
              type: string
              example: ok

        Product:
          type: object
          required:
            - id
            - name
            - price
            - currency
          properties:
            id:
              type: string
              format: uuid
            name:
              type: string
              example: Bio-Zitrone
            description:
              type: string
              example: Frische Bio-Zitrone aus nachhaltigem Anbau
            price:
              type: number
              format: double
              minimum: 0
              example: 1.95
            currency:
              type: string
              minLength: 3
              maxLength: 3
              example: CHF
            available:
              type: boolean
              example: true

        CreateOrderRequest:
          type: object
          required:
            - items
          properties:
            items:
              type: array
              minItems: 1
              items:
                $ref: '#/components/schemas/CreateOrderItem'

        CreateOrderItem:
          type: object
          required:
            - productId
            - quantity
          properties:
            productId:
              type: string
              format: uuid
            quantity:
              type: integer
              minimum: 1
              example: 2

        Order:
          type: object
          required:
            - id
            - status
            - items
            - total
            - currency
          properties:
            id:
              type: string
              format: uuid
            status:
              type: string
              enum:
                - pending
                - confirmed
                - paid
                - cancelled
            items:
              type: array
              items:
                $ref: '#/components/schemas/OrderItem'
            total:
              type: number
              format: double
              minimum: 0
            currency:
              type: string
              example: CHF

        OrderItem:
          type: object
          required:
            - productId
            - quantity
            - unitPrice
          properties:
            productId:
              type: string
              format: uuid
            quantity:
              type: integer
              minimum: 1
            unitPrice:
              type: number
              format: double
              minimum: 0

        CreatePaymentRequest:
          type: object
          required:
            - paymentMethod
          properties:
            paymentMethod:
              type: string
              enum:
                - credit-card
                - invoice
                - twint

        Payment:
          type: object
          required:
            - id
            - orderId
            - status
          properties:
            id:
              type: string
              format: uuid
            orderId:
              type: string
              format: uuid
            status:
              type: string
              enum:
                - pending
                - completed
                - failed
EOF